In [1]:
# Cell 1: Imports + Sample Data

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 200)

# --- Sample Movies Dataset (can be replaced by real CSV later) ---
movies_data = [
    (1, "Inception", "Action Sci-Fi Thriller", "dream within a dream mind-bending heist in the subconscious world"),
    (2, "Interstellar", "Adventure Sci-Fi Drama", "space travel through wormholes to save humanity and find new home"),
    (3, "The Dark Knight", "Action Crime Drama", "vigilante hero Batman faces chaotic villain Joker in Gotham"),
    (4, "The Matrix", "Action Sci-Fi", "hacker discovers reality is a simulation and joins rebellion"),
    (5, "Fight Club", "Drama Thriller", "an office worker forms an underground fight club as outlet for frustration"),
    (6, "The Social Network", "Drama Biography", "story of Facebook creation and conflict between friends and co-founders"),
    (7, "Avengers: Endgame", "Action Adventure Sci-Fi", "superheroes travel through time to undo catastrophic snap"),
    (8, "Titanic", "Romance Drama", "love story unfolds on doomed luxury ship that hits an iceberg"),
    (9, "The Wolf of Wall Street", "Biography Comedy Crime", "stockbroker rises and falls through fraud drugs and excess"),
    (10, "La La Land", "Romance Musical Drama", "aspiring actress and jazz musician pursue dreams and love in LA")
]

movies = pd.DataFrame(movies_data, columns=["movie_id", "title", "genres", "description"])

print("🎬 Movies dataset:")
display(movies)

# --- Sample User Ratings Dataset ---
ratings_data = [
    # user_id, movie_id, rating (1–5)
    (1, 1, 5), (1, 2, 5), (1, 4, 4), (1, 7, 5),
    (2, 1, 4), (2, 3, 5), (2, 4, 4), (2, 9, 5),
    (3, 2, 5), (3, 5, 4), (3, 8, 4), (3, 10, 5),
    (4, 3, 5), (4, 4, 4), (4, 6, 5), (4, 9, 4),
    (5, 7, 5), (5, 1, 4), (5, 8, 4), (5, 10, 4),
]

ratings = pd.DataFrame(ratings_data, columns=["user_id", "movie_id", "rating"])

print("\n👥 Ratings dataset:")
display(ratings.head())


🎬 Movies dataset:


,movie_id,title,genres,description
0,1,Inception,Action Sci-Fi Thriller,dream within a dream mind-bending heist in the subconscious world
1,2,Interstellar,Adventure Sci-Fi Drama,space travel through wormholes to save humanity and find new home
2,3,The Dark Knight,Action Crime Drama,vigilante hero Batman faces chaotic villain Joker in Gotham
3,4,The Matrix,Action Sci-Fi,hacker discovers reality is a simulation and joins rebellion
4,5,Fight Club,Drama Thriller,an office worker forms an underground fight club as outlet for frustration
5,6,The Social Network,Drama Biography,story of Facebook creation and conflict between friends and co-founders
6,7,Avengers: Endgame,Action Adventure Sci-Fi,superheroes travel through time to undo catastrophic snap
7,8,Titanic,Romance Drama,love story unfolds on doomed luxury ship that hits an iceberg
8,9,The Wolf of Wall Street,Biography Comedy Crime,stockbroker rises and falls through fraud drugs and excess
9,10,La La Land,Romance Musical Drama,aspiring actress and jazz musician pursue dreams and love in LA



👥 Ratings dataset:


,user_id,movie_id,rating
0,1,1,5
1,1,2,5
2,1,4,4
3,1,7,5
4,2,1,4


In [2]:
# Cell 2: Content-Based Recommendation (Movie-to-Movie)

# Combine genres + description into a single text feature
movies["combined_features"] = movies["genres"] + " " + movies["description"]

# Vectorize text
vectorizer = CountVectorizer(stop_words="english")
feature_matrix = vectorizer.fit_transform(movies["combined_features"])

# Compute cosine similarity between all movies
similarity_matrix = cosine_similarity(feature_matrix)

# Helper: get movie index by title
title_to_index = pd.Series(movies.index, index=movies["title"].str.lower())


def recommend_by_movie(title, top_n=5):
    """
    Recommend similar movies based on content (genres + description).
    """
    title_lower = title.lower()
    if title_lower not in title_to_index:
        print(f"❌ Movie '{title}' not found in database.")
        return

    idx = title_to_index[title_lower]
    sim_scores = list(enumerate(similarity_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1 : top_n + 1]  # skip itself

    print(f"\n🎥 Because you watched: **{movies.loc[idx, 'title']}**")
    print("👉 You might also like:\n")

    recs = []
    for i, score in sim_scores:
        recs.append({
            "title": movies.loc[i, "title"],
            "similarity_score": round(score, 3),
            "genres": movies.loc[i, "genres"]
        })

    recs_df = pd.DataFrame(recs)
    display(recs_df)


# Example
recommend_by_movie("Inception", top_n=5)



🎥 Because you watched: **Inception**
👉 You might also like:



,title,similarity_score,genres
0,The Matrix,0.277,Action Sci-Fi
1,Avengers: Endgame,0.263,Action Adventure Sci-Fi
2,Interstellar,0.167,Adventure Sci-Fi Drama
3,Fight Club,0.088,Drama Thriller
4,The Dark Knight,0.084,Action Crime Drama


In [3]:
# Cell 3: Simple Collaborative Filtering (User-to-Movie Recommendations)

# Create user-item rating matrix
user_item_matrix = ratings.pivot_table(index="user_id", columns="movie_id", values="rating")

# Replace NaN with 0 (unrated)
user_item_filled = user_item_matrix.fillna(0)

# Compute user similarity using cosine similarity
user_similarity = cosine_similarity(user_item_filled)
user_similarity_df = pd.DataFrame(user_similarity, index=user_item_filled.index, columns=user_item_filled.index)

print("👥 User similarity matrix:")
display(user_similarity_df.round(2))


def recommend_for_user(user_id, top_n=5):
    """
    Recommend movies for a given user based on similar users' ratings.
    """
    if user_id not in user_item_filled.index:
        print(f"❌ User ID {user_id} not found.")
        return

    # Similarity scores for this user
    sim_scores = user_similarity_df.loc[user_id]

    # Weighted average rating for each movie based on similar users
    weighted_scores = user_item_filled.T.dot(sim_scores) / (sim_scores.sum() + 1e-9)

    # Movies user has already rated
    already_rated = user_item_matrix.loc[user_id].dropna().index

    # Filter out already rated movies
    rec_scores = weighted_scores.drop(index=already_rated).sort_values(ascending=False)

    top_movies = rec_scores.head(top_n).index
    recs = movies[movies["movie_id"].isin(top_movies)].copy()
    recs["estimated_score"] = rec_scores.loc[top_movies].values.round(2)

    print(f"\n🎯 Recommendations for User {user_id}:")
    display(recs[["title", "genres", "estimated_score"]])


# Example
recommend_for_user(user_id=1, top_n=5)


👥 User similarity matrix:


user_id,1,2,3,4,5
user_id,,,,,
1,1.00,0.42,0.29,0.19,0.55
2,0.42,1.00,0.00,0.74,0.21
3,0.29,0.00,1.00,0.00,0.47
4,0.19,0.74,0.00,1.00,0.00
5,0.55,0.21,0.47,0.00,1.00



🎯 Recommendations for User 1:


,title,genres,estimated_score
2,The Dark Knight,Action Crime Drama,1.50
4,Fight Club,Drama Thriller,1.38
7,Titanic,Romance Drama,1.23
8,The Wolf of Wall Street,Biography Comedy Crime,1.16
9,La La Land,Romance Musical Drama,0.47


In [4]:
# Cell 4: Simple EDA

print("🎬 Number of movies:", len(movies))
print("👥 Number of ratings:", len(ratings))
print("👤 Unique users:", ratings['user_id'].nunique())

print("\n⭐ Average rating per movie:")
avg_ratings = ratings.groupby("movie_id")["rating"].mean().reset_index()
avg_ratings = avg_ratings.merge(movies[["movie_id","title"]], on="movie_id")
display(avg_ratings.sort_values(by="rating", ascending=False))


🎬 Number of movies: 10
👥 Number of ratings: 20
👤 Unique users: 5

⭐ Average rating per movie:


,movie_id,rating,title
1,2,5.000000,Interstellar
2,3,5.000000,The Dark Knight
6,7,5.000000,Avengers: Endgame
5,6,5.000000,The Social Network
9,10,4.500000,La La Land
8,9,4.500000,The Wolf of Wall Street
0,1,4.333333,Inception
3,4,4.000000,The Matrix
4,5,4.000000,Fight Club
7,8,4.000000,Titanic
